# Group Tasks, Week 4

* Please follow the guidelines for [submitting notebooks](https://fmfi-compbio.github.io/viz/Tutorial1.html\#pokyny-k-zadaniam) and for [working in teams](https://fmfi-compbio.github.io/viz/Groups.html). 
* Do not forget to switch off AI assistants in Colab `Menu->Tools->Settings->AI Assistance`. 
* Use only the libraries imported below. 
* To process data, use Pandas functions rather than loops or list comprehension.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
# setting for displaying 1 decimal number in Pandas tables
pd.options.display.float_format = '{:,.2f}'.format

## Task 1: mean and median


This task asks you to think about properties and definitions of mean and median.



### Task 1a (1 point)

We have unknown values $x=(x_1,x_2,x_3)$ and $y=(y_1,y_2,y_3,y_4)$ such that $\bar{x}=3$ and $\bar{y}=5$. How much is the mean of $z=(x_1,x_2,x_3,y_1,y_2,y_3,y_4)$? To the code cell below write a Python expression which will compute and print the answer.


In [ ]:
### TASK 1a START
### TASK 1a END

### Task 1b (3 points)

We have unknown values $x=(x_1,x_2,x_3)$ and $y=(y_1,y_2,y_3,y_4)$ such that
$x_1\le x_2\le x_3$ and $y_1\le y_2\le y_3\le y_4$ and median of $x$ is 3, and median of $y$ is 5. Now consider the median of $z=(x_1,x_2,x_3,y_1,y_2,y_3,y_4)$ (note that this sequence it not necessarily sorted, we can have $x_3>y_1$). The value of the median is not uniquely determined by the constraints above. Find some values of $x$ and $y$ so that the median of $z$ is as low as possible and some values of $x$ and $y$ so that the median of $z$ is as high as possible. 

Recall that the median of $x$ is $x_2$, the median of $y$ is $(y_2+y_3)/2$ and the median of $z$ is the middle element in the sorted order.

### TASK 1b START

Case where median of $z$ is as low as possible:

* x:
* y: 
* median of z:

Case where median of $z$ is as high as possible:

* x:
* y: 
* median of z:

### TASK 1b END



## Task 2: similarities between life expectancy time series

In this task, we return to the life expectancy time series from the Gapminder foundation. This data set gives life expectancy values for each country and for every year from 1900 to 2021. For a small selection of countries, we will compare all pairs of these time series and compute how similar or different they are.

### Importing life expectancy

Life expectancy data provided free by the [Gapminder foundation](https://www.gapminder.org/data/) and [World Bank](https://databank.worldbank.org/reports.aspx?source=2&series=SP.DYN.LE00.IN&country=#) under the CC-BY license.

After importing it, we select only 10 countries, transpose the table so that countries are columns and years are the rows, and rename the rows so that the years are integers rather than strings. Function [`transpose`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.transpose.html) switches rows and columns of a DataFrame.

We will work with the resulting DataFrame `life_exp`.

In [ ]:
url="https://fmfi-compbio.github.io/viz/data/life_expectancy_years.csv"
all_countries = pd.read_csv(url, index_col=0)
# drop column 0 with ISO
all_countries.drop(columns='ISO3', inplace=True)
# convert years from strings to ints 
all_countries.rename(columns=(lambda x : int(x)), inplace=True)

selected_countries = ['Saudi Arabia', 'Senegal', 'Serbia', 'Singapore',
                      'Slovak Republic', 'Slovenia', 'South Africa',
                      'Spain', 'Sri Lanka', 'Sweden']
# select only countries listed above and call transpose
life_exp = all_countries.loc[selected_countries, :].transpose()
display(Markdown("**The first two rows of `life_exp`**"))
display(life_exp.head(2))
display(Markdown("**Basic statistics of individual table columns (countries) obtained by `describe()`**"))
# print summary statistics, transposed for easier readability
life_exp.describe().transpose()


### Pearson correlation and its visualization

The command below computes Pearson correlation between all pairs of countries in our selection to table `corr` and visualizes it using [`sns.heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html).

In [ ]:
corr = life_exp.corr()
sns.heatmap(corr)
pass

### Task 2a: most and least correlated pairs (2 points)

Table `corr` has both rows and columns indexed by country names. Implement function `sort_table` which gets such a table and converts it from wide to long format so that it will have three columns named `country1`, `country2` and `value`, where `value` is the correlation for the two countries. 

Then filter the long table so that it contains only rows where `country1 < country2` (this will omit diagonal values and a redundant half of a symmetric heatmap). Finally sort the table by column `value` from smallest to largest. Return the resulting sorted table.

To convert from wide to long table, you can use function `melt` similarly as it was used in [lecture 03](https://colab.research.google.com/github/fmfi-compbio/viz/blob/master/notebooks/L03_Plot_types.ipynb). Before `melt`, we recommend calling `reset_index`. You may also  need functions `rename`, `query` and `sort_values`.

You can assume that both column and row indices are called `Country`, but otherwise your function should work also for tables that differ from `corr` in their values or in the sets of countries.

In [ ]:
def sort_table(corr_table):  
  ### TASK 2a START
  ### TASK 2a END


# calling your function
sorted = sort_table(corr)

display(Markdown("**Running some checks**"))
assert sorted.shape == (45,3)
assert list(sorted.columns) == ['country1', 'country2', 'value']
assert sorted.iloc[-3, 0] == 'Spain'
assert sorted.iloc[-3, 1] == 'Sweden'
assert abs(sorted.iloc[-3, 2] - 0.987) < 0.001
print("OK")

# printing the most and least correlated pairs
display(Markdown("**Pairs with highest correlations:**"))
display(sorted.tail().iloc[::-1])
display(Markdown("**Pairs with lowest correlations:**"))
display(sorted.head())


### Task 2b: computing Euclidean distance (1 point)

Perhaps you have noticed that all pairs of countries have high correlation coefficients. This is because the life expectancy generally has an increasing tendency in all countries. Pearson correlation can be very high even if values for one country are consistently lower than for the other country, because it does not change with linear tranformation $ax_i+b$. 

For these reasons, perhaps a better measure for comparing these time series is the Euclidean distance which compares $x_1,\dots, x_n$ and $y_1,\dots y_n$ using formula:

$$ \sqrt{\sum_{i=1}^n(x_i-y_i)^2}$$

Function `dist` below computes this distance. Your task has three easy parts:
* Call function `corr` on `life_exp` with argument `method=dist` where `dist` is the function below. This will compute the table of Euclidean distances between all pairs of countries; store it in a variable.
* Visualize the distance table using heatmap as in above.
* Call `sort_table` on the distance table, storing the result in `sorted2`. 

The code below then prints countries with the smallest distances (most similar) and with the highest distances (least similar).

In [ ]:
def dist(x, y):
  return np.linalg.norm(x-y)

### TASK 2b START
# ...
# sorted2 = 
### TASK 2b END

display(Markdown("**Pairs with lowest distances:**"))
display(sorted2.head())
display(Markdown("**Pairs with highest distances:**"))
display(sorted2.tail().iloc[::-1])


### Task 2c: visualizing selected pairs of countries (3 points)

Your task here is to write function `draw_pair`, which gets DataFrame `life_exp` as defined above and the names of two countries from this table and draws two plots. 
* The first plot should contain life expectancies of these two countries as a line graph, each country having a line with a different color (use the index of the table as x coordinate). Do not forget to include a legend.
* The second plot should be a scatter plot with a regression line ([`sns.regplot`](https://seaborn.pydata.org/generated/seaborn.regplot.html)), where you plot each year as one dot, with expectation in the first country as x coordinate and in the second country as y coordinate. 
The title of the figure should contain the names of the countries, and the Pearson correlation and Euclidean distance of these two countries (compute them by calling `corr` and `dist` on two table columns). The plot thus should look as [this example](https://fmfi-compbio.github.io/viz/img/G04-t2d.png), although the exact style may differ.

The code below then calls this function for the least similar pairs of countries: first those with the smallest correlation and then for those with the largest distance. Observe the differences, but you do not need to comment on them.

In [ ]:
def draw_pair(table, country1, country2):  
  figure, axes = plt.subplots(1, 2, figsize=(12, 4))
  ### TASK 2c START
  
  ### TASK 2c END
  

# lowest correlation
draw_pair(life_exp, "Senegal", "South Africa")
# largest distance
draw_pair(life_exp, "Senegal", "Sweden")

## Task Bonus: IRQ and standard deviation

Use table `all_countries`. For each year in this table, compute quartiles Q1 and Q3 among all countries. Also compute IQR (interquantile range) and the standard deviation for each year. Create line plots of these quantities and comment on them. 

In [ ]:
### TASK bonus START
# your code here
### TASK bonus END


### TASK bonus START
Your commentary on your plots here (in Slovak or English).
### TASK bonus END